$\textbf{Notes for ME 263 Lecture - 2}\\
\text{Submitted by - Shivansh Srivastava}\\
\text{SR No. - 27240}
$

$\text{In this lecture, we will first derive the weak form of the Poisson's equation.}\\
\text{To begin with, we first define the boundary value problem as follows:}
$
$$
\begin{cases}
-\Delta u = f & \text{in } \Omega \tag{1}, \\[6pt]
u = g & \text{on } \Gamma_D, \\[6pt]
-\nabla u \cdot n = \tau & \text{on } \Gamma_N.
\end{cases}
$$


$\text{To get the weak form of the above PDE, we first multiply both sides of the governing equation by a test function v}$ 
$\text{and then we integrate the resulting equations over the entire domain} $ $\Omega$. \
$\text{This gives}$
$$
-\int_{\Omega} v\,\Delta u
=
\int_{\Omega} vf,
\qquad \forall v \in V
$$
$\text{Integrating the above expression by parts gives}$
$$
\int_{\Omega} \nabla u \cdot \nabla v
-
\int_{\Omega} \nabla \cdot (v\nabla u)
=
\int_{\Omega} vf \tag{2}
$$
$\text{Now, we use the Gauss divergence theorem on the second term in the above expression to convert the volume }$
$\text {integral into a surface integral, which gives,}$
$$
\int_{\Omega} \nabla u \cdot \nabla v
-
\int_{\Gamma} v\,\nabla u\cdot n
=
\int_{\Omega} vf \tag{3}
$$
$\text{The surface integral in (3) can be further decomposed into 2 parts, which can be easily simplified based on the given}$ 
$\text{boundary conditions.}$\
$\text{(Note that v = 0 on the portion of the surface where dirichlet boundary condition is specified.)}$

$$
\int_{\Omega} \nabla u \cdot \nabla v
-
\int_{\Gamma_N} v\,\nabla u\cdot n
-
\int_{\Gamma_D} v\,\nabla u\cdot n
=
\int_{\Omega} vf
$$
$\text{This gives}$
$$
\int_{\Omega} \nabla u \cdot \nabla v
+
\int_{\Gamma_N} \tau\,v
=
\int_{\Omega} fv
$$
$\text{Rearranging the above expression gives us the final weak form as}$
$$
\int_{\Omega} \nabla u \cdot \nabla v
=
\int_{\Omega} fv
-
\int_{\Gamma_N} \tau\,v \tag{4}
$$
$\text{which is in the standard form of}$

$$
a(u,v) = L(v), \qquad \forall \qquad v \in V
$$

$\text{There are certain properties that our set V of all possible variations (v) must satisfy, and these are given as: }$ 

$$
\begin{cases}
\text{v must be square integrable in V,} \\[6pt]
\nabla v \text{ must be square integrable in V,} \\[6pt]
\text{v = 0 on } \Gamma_D \\[6pt]
\end{cases}
$$
$\text{These conditions can also be expressed as}$
$$
\left\{
\begin{array}{ll}
v \in H^1(\Omega) & \text{s.t. } v = 0 \text{ on } \Gamma_D
\end{array}
\right\}
$$

$\text{Next, we will see how we can query the output of the code that we saw in the last class (Lecture - 1).}$
$\text{Please note that the cells below need to be copied into the notebook for lecture 1, so that the variables take up their}$
 $\text{computed values.}$

In [ ]:
print("type of u:", type(u), "number = ", u.number()) 
print("type of v:", type(v), "number = ", v.number())

$\text{This will give the output as: }$ 

type of u: <class 'ufl.argument.Argument'> number = 1 \
type of v: <class 'ufl.argument.Argument'> number = 0 


$\text{This tells that u and v are UFL Arguments. These are symbolic placeholders representing the unknown FE}$
$\text{functions. Moreover, u.number() returns 1 and v.number() represents 0 because it is UFL's way of distinguishing}$
$\text{different arguments}$

In [ ]:
print("type of a:", type(a), "arguments of a: ", sorted(q.number() for q in a.arguments()))

$\text{This will give the output as: }$ 

type of a: <class 'ufl.form.Form'> arguments of a: [0, 1]

$\text{There are 2 important pieces of information here.}$ 

$\text{Firstly, a.type() tells us that a is a UFL form. It is not a matrix yet.}$ \
$\text{Only when we will do A = assemble\_matrix(form(a)) (discussed later), will the matrix be assembled.}$
$\text{Moreover, since a takes 2 arguments (u and v), a.arguments(), when sorted returns [0, 1], as the numbers}$
$\text{associated with these are 1 and 0 respectively.}$

In [ ]:
print("type of L:", type(L))

$\text{This will give the output as: }$ 

type of L: <class 'ufl.form.Form'>

$\text{Note that L is also of the type form, similar to a.}$

In [ ]:
print("arguments of L:", sorted(q.number() for q in L.arguments()))

$\text{This will give the output as: }$ 

arguments of L: [0]

$\text{Since L takes only one argument (v), L.arguments() returns 0, as the number associated with v is 0.}$

In [ ]:
#form is needed to wrap the UFL expressions into a form that dolfinx can use for FE assembly 
from dolfinx.fem import form 
#assemble_matrix and assemble_vector are needed to convert the weak forms into PETSc objects.
from dolfinx.fem.petsc import assemble_matrix, assemble_vector

In [ ]:
A = assemble_matrix(form(a), bcs=[bc]); A.assemble()

$\text{Here, assemble\_matrix populates the FE matrix, whereas bcs=[bc] imposes the specified boundary conditions.}$
$\text{Moreover, A.assemble() is called to finalize the construction of the PETSc matrix which might be distributed}$ 
$\text{on multiple processors.}$

In [ ]:
print("A - > ", type(A).__name__, "size = ", A.getSize())

$\text{This will print the type of A as Matrix (using the dunder method \_\_name\_\_.)}$
$\text{A.getsize() will return the size of the final A matrix.}$

In [ ]:
bvec = assemble_vector(form(L))

$\text{Now we assemble the right hand side using assemble\_vector()}$

In [ ]:
print("bvec - > ", type(bvec).__name__, "size = ", bvec.getLocalSize())

$\text{Here, we will get the type of the RHS as vector.}$
$\text{Moreover, getLocalSize() will return the number of entities owned by the current MPI process, instead of }$ 
$\text{the global DOFs}$